# PaddleOCR vs EasyOCR on our labelled crops

Brief §11.3: keep EasyOCR only if it beats PaddleOCR by **≥ 2 points** of exact-plate accuracy on the same held-out photos. Until this is run, the worker uses PaddleOCR only (docs/decisions.md D18).

Data: `anpr/eval/data/<set>/` as described in `anpr/DATASET.md`.

In [ ]:
%pip install -q easyocr
import csv, re, cv2
from pathlib import Path
from anpr.config import load_settings
from anpr.detector import PlateDetector
from anpr.ocr import PaddleOcrEngine, Line, read_plate

In [ ]:
import easyocr


class EasyOcrEngine:
    def __init__(self):
        self.reader = easyocr.Reader(["en"], gpu=False, verbose=False)

    def read_lines(self, image):
        out = []
        for box, text, conf in self.reader.readtext(
            image, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
        ):
            xs = [p[0] for p in box]
            ys = [p[1] for p in box]
            out.append(Line(text, float(conf), min(ys), min(xs), max(ys) - min(ys) or 1.0))
        return out

In [ ]:
SET = Path("../eval/data/my-set")
rows = list(csv.DictReader((SET / "labels.csv").open()))
det = PlateDetector(load_settings().detector_model)
engines = {"paddle": PaddleOcrEngine(), "easyocr": EasyOcrEngine()}
score = {k: 0 for k in engines}
for row in rows:
    img = cv2.imread(str(SET / row["filename"]))
    boxes = det.detect(img)
    if not boxes:
        continue
    truth = re.sub("[^A-Z0-9]", "", row["plate"].upper())
    for name, eng in engines.items():
        r = read_plate(eng, img, boxes[0])
        score[name] += bool(r and r.text == truth)
for name, s in score.items():
    print(f"{name}: {s}/{len(rows)} = {s / len(rows):.1%}")